<a href="https://colab.research.google.com/github/don-jose07/E-commerce-Sales-Analysis-/blob/main/Student_Grade_Management_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Student Grade Management System (OOP Python Mini Project)

This project aims to create a menu-driven Student Grade Management System using Object-Oriented Programming (OOP) in Python. It will demonstrate the use of classes, objects, methods, data structures, exception handling, and file handling.

We will start by defining the `Student` class, which will represent individual students and their academic records.

In [ ]:
class Student:
    def __init__(self, student_id, name):
        if not isinstance(student_id, str) or not student_id:
            raise ValueError("Student ID must be a non-empty string.")
        if not isinstance(name, str) or not name:
            raise ValueError("Student name must be a non-empty string.")

        self.student_id = student_id
        self.name = name
        self.grades = {}

    def add_grade(self, subject, grade):
        if not isinstance(subject, str) or not subject:
            raise ValueError("Subject must be a non-empty string.")
        if not isinstance(grade, (int, float)) or not (0 <= grade <= 100):
            raise ValueError("Grade must be a number between 0 and 100.")

        self.grades[subject] = grade
        print(f"Grade {grade} added for {self.name} in {subject}.")

    def get_average_grade(self):
        if not self.grades:
            return 0.0
        return sum(self.grades.values()) / len(self.grades)

    def display_info(self):
        info = f"\nStudent ID: {self.student_id}\nName: {self.name}\nGrades: {self.grades}"
        if self.grades:
            info += f"\nAverage Grade: {self.get_average_grade():.2f}"
        else:
            info += "\nNo grades recorded yet."
        print(info)

    def to_dict(self):
        """Converts the student object to a dictionary for easy saving."""
        return {
            "student_id": self.student_id,
            "name": self.name,
            "grades": self.grades
        }

    @classmethod
    def from_dict(cls, data):
        """Creates a Student object from a dictionary."""
        student = cls(data['student_id'], data['name'])
        student.grades = data['grades']
        return student

# --- Demo of Student Class (for testing purposes) ---
print("\n--- Demonstrating Student Class ---")
try:
    student1 = Student("S001", "Alice Smith")
    student1.add_grade("Math", 85)
    student1.add_grade("Science", 92.5)
    student1.display_info()

    student2 = Student("S002", "Bob Johnson")
    student2.display_info() # No grades yet
    student2.add_grade("History", 78)
    student2.display_info()

    # Test invalid input
    # student3 = Student("", "Charlie") # This would raise a ValueError
    # student1.add_grade("English", 105) # This would raise a ValueError
except ValueError as e:
    print(f"Error creating student or adding grade: {e}")

## GradeManager Class

Now we will implement the `GradeManager` class. This class will serve as the central hub for our system, allowing us to manage a collection of `Student` objects. It will include methods for adding new students, removing existing ones, searching for students by ID or name, and displaying a list of all enrolled students. Later, we'll integrate file handling into this class.

In [ ]:
import json
import os

class GradeManager:
    def __init__(self, filename="students.json"):
        self.students = {}
        self.filename = filename
        self._load_students()

    def _load_students(self):
        if os.path.exists(self.filename):
            try:
                with open(self.filename, 'r') as f:
                    data = json.load(f)
                    for student_data in data:
                        student = Student.from_dict(student_data)
                        self.students[student.student_id] = student
                print(f"Loaded {len(self.students)} students from {self.filename}")
            except json.JSONDecodeError:
                print(f"Warning: {self.filename} is empty or contains invalid JSON. Starting with empty student list.")
            except Exception as e:
                print(f"Error loading students: {e}")
        else:
            print("No existing student data file found. Starting with an empty student list.")

    def _save_students(self):
        try:
            with open(self.filename, 'w') as f:
                json.dump([s.to_dict() for s in self.students.values()], f, indent=4)
            print(f"Saved {len(self.students)} students to {self.filename}")
        except Exception as e:
            print(f"Error saving students: {e}")

    def add_student(self, student_id, name):
        if student_id in self.students:
            print(f"Error: Student with ID {student_id} already exists.")
            return False
        try:
            new_student = Student(student_id, name)
            self.students[student_id] = new_student
            self._save_students()
            print(f"Student {name} (ID: {student_id}) added successfully.")
            return True
        except ValueError as e:
            print(f"Error adding student: {e}")
            return False

    def remove_student(self, student_id):
        if student_id not in self.students:
            print(f"Error: Student with ID {student_id} not found.")
            return False

        del self.students[student_id]
        self._save_students()
        print(f"Student with ID {student_id} removed successfully.")
        return True

    def find_student(self, search_term):
        results = []
        for student in self.students.values():
            if search_term.lower() in student.name.lower() or search_term.lower() == student.student_id.lower():
                results.append(student)
        return results

    def add_grade_to_student(self, student_id, subject, grade):
        student = self.students.get(student_id)
        if not student:
            print(f"Error: Student with ID {student_id} not found.")
            return False
        try:
            student.add_grade(subject, grade)
            self._save_students()
            return True
        except ValueError as e:
            print(f"Error adding grade: {e}")
            return False

    def display_all_students(self):
        if not self.students:
            print("No students in the system.")
            return
        print("\n--- All Students ---")
        for student in self.students.values():
            student.display_info()
            print("--------------------")


# --- Demo of GradeManager Class (for testing purposes) ---
print("\n--- Demonstrating GradeManager Class ---")
# Clean up any old file for a fresh demo
try:
    if os.path.exists("students.json"):
        os.remove("students.json")
        print("Cleaned up old students.json for demo.")
except Exception as e:
    print(f"Error cleaning up old file: {e}")

manager = GradeManager()

manager.add_student("S001", "Alice Smith")
manager.add_student("S002", "Bob Johnson")
manager.add_student("S003", "Charlie Brown")

manager.add_grade_to_student("S001", "Math", 88)
manager.add_grade_to_student("S001", "Science", 91)
manager.add_grade_to_student("S002", "History", 75)
manager.add_grade_to_student("S002", "Geography", 82.5)

manager.display_all_students()

print("\n--- Searching for 'Alice' ---")
found_students = manager.find_student("Alice")
if found_students:
    for student in found_students:
        student.display_info()
else:
    print("No students found matching 'Alice'.")

print("\n--- Removing S003 ---")
manager.remove_student("S003")
manager.display_all_students()

print("\n--- Trying to add existing student S001 ---")
manager.add_student("S001", "Alice Smith Again") # Should fail

print("\n--- Trying to add grade to non-existent student S004 ---")
manager.add_grade_to_student("S004", "Physics", 90) # Should fail

print("\n--- Creating a new manager to load saved data ---")
manager2 = GradeManager() # Should load S001 and S002
manager2.display_all_students()

## Menu-Driven Interface (Main Program)

Finally, we will create the menu-driven interface to interact with our `GradeManager`. This will be the main entry point of our application, providing options for:

1.  Add New Student
2.  Add Grade to Student
3.  Remove Student
4.  Search Student
5.  Display All Students
6.  Exit

The interface will handle user input, call the appropriate `GradeManager` methods, and provide feedback to the user, including error handling for invalid inputs.

In [ ]:
def display_menu():
    print("\n--- Student Grade Management System ---")
    print("1. Add New Student")
    print("2. Add Grade to Student")
    print("3. Remove Student")
    print("4. Search Students")
    print("5. Display All Students")
    print("6. Exit")
    print("---------------------------------------")

def main():
    manager = GradeManager()

    while True:
        display_menu()
        choice = input("Enter your choice: ").strip()

        if choice == '1':
            student_id = input("Enter student ID: ").strip()
            name = input("Enter student name: ").strip()
            manager.add_student(student_id, name)

        elif choice == '2':
            student_id = input("Enter student ID to add grade: ").strip()
            subject = input("Enter subject: ").strip()
            try:
                grade = float(input("Enter grade (0-100): ").strip())
                manager.add_grade_to_student(student_id, subject, grade)
            except ValueError:
                print("Invalid grade. Please enter a number between 0 and 100.")

        elif choice == '3':
            student_id = input("Enter student ID to remove: ").strip()
            manager.remove_student(student_id)

        elif choice == '4':
            search_term = input("Enter student ID or name to search: ").strip()
            results = manager.find_student(search_term)
            if results:
                print("\n--- Search Results ---")
                for student in results:
                    student.display_info()
                    print("--------------------")
            else:
                print("No students found matching your search term.")

        elif choice == '5':
            manager.display_all_students()

        elif choice == '6':
            print("Exiting Student Grade Management System. Goodbye!")
            break

        else:
            print("Invalid choice. Please enter a number between 1 and 6.")

if __name__ == "__main__":
    # Optionally clean up the students.json file if you want to start fresh every run
    # import os
    # if os.path.exists("students.json"):
    #     os.remove("students.json")
    main()

Loaded 2 students from students.json

--- Student Grade Management System ---
1. Add New Student
2. Add Grade to Student
3. Remove Student
4. Search Students
5. Display All Students
6. Exit
---------------------------------------
